# Lesson 4.5: Prove it: same trajectory, same score

**Goal:** show that the `create_agent` rebuild of Nightwatch (lesson 4.4) runs the same loop as v0, by feeding both the
same recorded model replies, and read a live 10-incident comparison without fooling yourself.

**What you will be able to do**
- Record v0's raw replies with a spy and play them back through a scripted chat model.
- Compare two runs by trajectory (tool calls + arguments, writes, finish), not only by grade.
- Explain the 5 replay differences by counting steps by hand.
- Tell a real difference from sampling noise at n = 10.

In [1]:
# Setup: find the course repo's nightwatch/ folder (relative to this notebook) and load .env if present.
import os, sys
from pathlib import Path

for up in ("../../nightwatch", "../../../nightwatch", "../nightwatch", "nightwatch"):
    if Path(up, "brightcart").is_dir():
        NW = Path(up)
        break
else:
    raise FileNotFoundError("Open this notebook from <lesson>/notebook inside the course repo: it needs the nightwatch/ folder.")
sys.path.insert(0, str(NW.resolve()))

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))      # walks up from here and loads the repo's .env
except ImportError:
    pass
HAS_KEY = bool(os.getenv("GROQ_API_KEY"))
print("nightwatch folder:", NW.as_posix(), "| GROQ_API_KEY:", "found" if HAS_KEY else "not set")
if not HAS_KEY:
    print("No GROQ_API_KEY: get the free key as shown in lesson 0.2, put GROQ_API_KEY=... in the repo's .env,")
    print("then restart. Cells that need it: only the live cell in part 4 (it replays from the record cache, so a re-run costs nothing). They replay from the cache when it has the answer.")

nightwatch folder: ../../../nightwatch | GROQ_API_KEY: found


## 1. Recall: two pieces of middleware (lesson 4.4)

When Groq rejected a malformed tool call, **`retry_malformed`** told the model and called it again. v0's 14-step loop became
**`ModelCallLimitMiddleware(run_limit=14)`**. Here is the middleware list of the rebuild (building it calls nothing):

In [2]:
import json
sys.path.insert(0, str(Path("../../l4_04_create_agent/examples").resolve()))
import nw_agent                       # lesson 4.4's create_agent rebuild of Nightwatch
from nw_agent import agent_v0         # Nightwatch v0 (chapter 3)

box = {"final": None, "trace": [], "errors": []}
for m in nw_agent.make_middleware(box):
    print(getattr(m, "name", type(m).__name__))

nudge_to_tools
ModelCallLimitMiddleware
retry_malformed
record_replay
log_tool


## 2. Key words

| Word | Meaning here |
|---|---|
| **trajectory** | the path of one run: each tool call with its arguments, the writes, and the finish |
| **scripted model** | a fake chat model that plays back a fixed list of replies, in order |
| **exact replay** | a recorded run that, played back from the cache, gives the very same steps and writes |
| **sampling** | at temperature 1, each live run is a fresh random draw from the model |

## 3. The idea: hold the replies fixed

A trajectory comes from two things: **your loop code** and **the model's replies**. Two live runs mix both causes.
Record v0's replies, play the very same replies to `create_agent`, and only the code can differ.

```
v0 (replay from cache) --spy--> replies [r1 .. r14] --> Scripted(script=[...]) --> create_agent
        |                                                                          |
        +----------------- compare: tool calls + args, writes, finish, grade ------+
```

**Hand trace: why `memory_leak-5`, sample 2 differs.** v0 got 14 replies; reply 3 was malformed.

| | v0 | create_agent |
|---|---|---|
| how a malformed reply counts | one of its 14 steps | retried *inside* one model call |
| 14 replies count as | 13 good + 1 malformed = 14 steps | 13 model calls |
| at the limit | stops: step limit | under the limit of 14, asks for reply 15 |
| result | `stop: step limit` | the script is empty: **script ran out** |

Same loop, same replies: the two versions only *count* a retry differently.

## 4. Write it yourself

### 4a. The baseline runs and the scripted model

In [3]:
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from brightcart.incidents import TEMPLATES, generate

incs = {x.id: x for x in generate()}
rows = [json.loads(line) for line in open(NW / "runs" / "baseline_gpt-oss-20b.jsonl", encoding="utf-8")]
print(len(rows), "baseline runs")

class Scripted(BaseChatModel):
    """A chat model that plays back a fixed list of replies (or a Malformed error), in order."""
    script: list

    def _generate(self, messages, stop=None, run_manager=None, **kw):
        if not self.script:
            raise IndexError("script ran out: create_agent asked for more model calls than v0 made")
        item = self.script.pop(0)
        if item == "MALFORMED":
            raise nw_agent.Malformed("scripted")      # same error Groq's HTTP 400 raises in our agent
        return ChatResult(generations=[ChatGeneration(message=item)])

    def bind_tools(self, tools, **kw):                # create_agent attaches tools; a list does not care
        return self

    @property
    def _llm_type(self):
        return "scripted"

180 baseline runs


### 4b. A spy on v0's chat, and two helpers

In [4]:
def v0_with_replies(inc, sample, mode="replay"):
    """Run agent_v0 and keep every raw model reply it got."""
    replies, real = [], agent_v0.chat
    def spy(body, mode, sample):
        out = real(body, mode, sample)
        replies.append(out)
        return out
    agent_v0.chat = spy
    try:
        r = agent_v0.run(inc, mode=mode, sample=sample)
    finally:
        agent_v0.chat = real                          # always put the real chat back
    return r, replies

def to_message(out):
    """A saved reply -> AIMessage, an error reply -> the marker MALFORMED."""
    if "error" in out:
        return "MALFORMED"
    m = out["choices"][0]["message"]
    calls = []
    for c in m.get("tool_calls") or []:
        try:
            args = json.loads(c["function"]["arguments"] or "{}")
        except json.JSONDecodeError:
            args = {}
        calls.append({"name": c["function"]["name"], "args": args, "id": c["id"], "type": "tool_call"})
    return AIMessage(content=m.get("content") or "", tool_calls=calls)

def seq(r):
    """The tool calls of a run, with their arguments."""
    return [(t.get("tool"), json.dumps(t.get("args"), sort_keys=True)) for t in r["trace"] if t.get("tool")]

v0, replies = v0_with_replies(incs["memory_leak-5"], 2)
print("memory_leak-5 s2: v0 steps", v0["steps"], "| stop", v0["stop_reason"], "| replies", len(replies),
      "| malformed at", [i + 1 for i, o in enumerate(replies) if "error" in o])

memory_leak-5 s2: v0 steps 14 | stop step limit | replies 14 | malformed at [3]


### 4c. The proof loop (no API calls: v0 replays from its cache, create_agent reads a list)
Only runs that replay exactly are used. How many will match? Guess before you run it.

In [5]:
def same_traj(a, b):
    return a is not None and seq(a) == seq(b) and a["writes"] == b["writes"] and a["final"] == b["final"]

def exact_replays():
    """(incident, sample, v0 run, replies) for every baseline run that replays exactly."""
    out = []
    for row in rows:
        inc = incs[row["incident"]]
        try:
            v0, replies = v0_with_replies(inc, row["sample"])
        except LookupError:
            continue                                   # a reply is missing from the cache
        if v0["steps"] == row["steps"] and v0["writes"] == [tuple(w) for w in row["writes"]]:
            out.append((inc, row["sample"], v0, replies))
    return out

def prove(run_ca, show=True):
    same_t = same_g = 0
    for inc, s, v0, replies in EXACT:
        try:
            ca, err = run_ca(inc, Scripted(script=[to_message(o) for o in replies])), ""
        except IndexError as e:
            ca, err = None, str(e)
        t_ok = same_traj(ca, v0)
        same_t += t_ok
        same_g += ca is not None and agent_v0.grade(ca) == agent_v0.grade(v0)
        if show and not t_ok:
            print(f"  differs: {inc.id} s{s} v0 steps {v0['steps']} stop {v0['stop_reason']} "
                  f"malformed replies {sum('error' in t for t in v0['trace'])} | create_agent: {err.split(':')[0] or ca['steps']}")
    return len(EXACT), same_t, same_g

EXACT = exact_replays()
n, same_t, same_g = prove(lambda inc, model: nw_agent.run(inc, model=model))
print(f"exact-replay runs: {n} | identical trajectory {same_t}/{n} | same grade {same_g}/{n}")

  differs: memory_leak-5 s2 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out
  differs: traffic_spike-0 s0 v0 steps 14 stop step limit malformed replies 2 | create_agent: script ran out


  differs: traffic_spike-1 s0 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out
  differs: traffic_spike-1 s2 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out


  differs: stuck_consumer-5 s1 v0 steps 14 stop step limit malformed replies 3 | create_agent: script ran out


exact-replay runs: 39 | identical trajectory 34/39 | same grade 34/39


**34 of 39** identical, and the same 34 got the same grade. All 5 differences hit v0's 14-step limit with at least one
malformed reply (1, 2, 1, 1, 3), and each time `create_agent` asked for more replies than v0 had made: the script ran out.
The honest limit: this covers the 39 runs that replay exactly, not all 180.

## 5. Library version + check

In [6]:
import importlib.metadata as md_
for p in ("langchain", "langgraph", "langchain-core", "langchain-groq"):
    print(f"{p:15}", md_.version(p))
assert (n, same_t, same_g) == (39, 34, 34), (n, same_t, same_g)
print("check: same numbers as the video (39 / 34 / 34)")

langchain       1.5.0
langgraph       1.2.22
langchain-core  1.8.1
langchain-groq  1.2.0
check: same numbers as the video (39 / 34 / 34)


## 6. Live: ten incidents, one run each (calls the API)

`gpt-oss-20b` at temperature 1, the 10 training incidents of lesson 3.6, once per version. v0 runs in record mode and
`create_agent` with its own record/replay cache, so after the first run this cell replays for free.

In [7]:
# calls the API (only on a cache miss; the recorded run replays for free)
TEN = [next(x for x in incs.values() if x.template == t and x.split == "train") for t in TEMPLATES]
jsonl = {(r["incident"], r["sample"]): r for r in rows}
res = []
if HAS_KEY:
    for inc in TEN:
        v0, _ = v0_with_replies(inc, 0, mode="record")
        ca = nw_agent.run(inc, sample=0)
        res.append((inc, v0, ca))
        gv, gc = agent_v0.grade(v0)["pass"], agent_v0.grade(ca)["pass"]
        print(f"{inc.id:18} jsonl {'PASS' if jsonl[(inc.id, 0)]['pass'] else 'fail'} | v0 {'PASS' if gv else 'fail'} "
              f"steps {v0['steps']:2} | create_agent {'PASS' if gc else 'fail'} steps {ca['steps']:2} retries {len(ca['errors'])}")
else:
    print("skipped: no GROQ_API_KEY (see the setup cell)")

bad_deploy-2       jsonl PASS | v0 PASS steps  9 | create_agent PASS steps  7 retries 5
upstream_deploy-1  jsonl PASS | v0 PASS steps  8 | create_agent PASS steps  8 retries 2


memory_leak-0      jsonl fail | v0 PASS steps 13 | create_agent PASS steps  9 retries 2
traffic_spike-0    jsonl fail | v0 fail steps 14 | create_agent PASS steps 10 retries 3


bad_flag-1         jsonl PASS | v0 PASS steps  7 | create_agent PASS steps  7 retries 1


stuck_consumer-1   jsonl fail | v0 fail steps 14 | create_agent fail steps 14 retries 1
expired_secret-1   jsonl PASS | v0 PASS steps  6 | create_agent PASS steps  7 retries 0


pool_config-2      jsonl PASS | v0 PASS steps  6 | create_agent PASS steps  8 retries 3
disk_full-2        jsonl fail | v0 fail steps 14 | create_agent PASS steps 10 retries 1


false_alarm-0      jsonl PASS | v0 PASS steps  3 | create_agent PASS steps  6 retries 0


In [8]:
if res:
    pv = sum(agent_v0.grade(v)["pass"] for _, v, _ in res)
    pc = sum(agent_v0.grade(c)["pass"] for _, _, c in res)
    pj = sum(jsonl[(i.id, 0)]["pass"] for i, _, _ in res)
    print(f"pass: baseline jsonl {pj}/10 | v0 now {pv}/10 | create_agent {pc}/10")
    print(f"identical tool sequences {sum(seq(v) == seq(c) for _, v, c in res)}/10 | "
          f"same first call (tool + args) {sum(seq(v)[:1] == seq(c)[:1] for _, v, c in res)}/10 | "
          f"same first tool by name {sum(seq(v)[:1][0][0] == seq(c)[:1][0][0] for _, v, c in res)}/10 | "
          f"same writes {sum(v['writes'] == c['writes'] for _, v, c in res)}/10")
    tv = sum(v["usage"]["prompt"] + v["usage"]["completion"] for _, v, _ in res)
    tc = sum(c["usage"]["prompt"] + c["usage"]["completion"] for _, _, c in res)
    print(f"tokens: v0 {tv:,} | create_agent {tc:,} | create_agent retries {sum(len(c['errors']) for _, _, c in res)}")
    for inc, v, c in res:
        if inc.id == "bad_flag-1":
            print("bad_flag-1 v0:", [t for t, _ in seq(v)])
            print("bad_flag-1 ca:", [t for t, _ in seq(c)])

pass: baseline jsonl 6/10 | v0 now 7/10 | create_agent 9/10
identical tool sequences 0/10 | same first call (tool + args) 4/10 | same first tool by name 8/10 | same writes 6/10
tokens: v0 91,465 | create_agent 101,886 | create_agent retries 18
bad_flag-1 v0: ['check_health', 'read_logs', 'list_flags', 'set_flag', 'check_health', 'finish']
bad_flag-1 ca: ['check_health', 'read_logs', 'list_flags', 'set_flag', 'check_health', 'check_health', 'finish']


Compare with v0's own record cache: how often was a reply malformed there? (no API, just counts files)

In [9]:
cache = list((NW / "runs" / "cache").glob("*.json"))
bad = sum("error" in json.loads(f.read_text(encoding="utf-8")) for f in cache)
print(f"v0 cache: {bad} of {len(cache):,} replies malformed ({bad / len(cache):.1%})")

v0 cache: 56 of 1,891 replies malformed (3.0%)


Nine against seven looks like a win, but v0 alone moved from 6 (baseline file) to 7 with no code change. With 10 incidents
and one run each, a single flip moves the score by 10 points. The retry-rate gap (18 retries in 10 live runs against about
3% in v0's cache) is still unexplained: same model, same tools, and we do not guess.

## 7. Where this shows up in agents: a regression test for your wiring

A scripted replay tests your agent's **wiring**, not the model. Upgrade LangChain or move the loop into a graph, re-run the
39, and any changed trajectory shows up at once, with no key and no tokens. Here it is as a test you could put in CI
(a changed model or prompt is a different question, and that needs many live runs per incident):

In [10]:
def test_wiring(k=5):
    """The first k exact replays that matched must still match."""
    for inc, s, v0, replies in EXACT[:k]:
        ca = nw_agent.run(inc, model=Scripted(script=[to_message(o) for o in replies]))
        assert same_traj(ca, v0), f"{inc.id} s{s} changed trajectory"
    return f"wiring OK on {k} replays"

print(test_wiring())

wiring OK on 5 replays


## 8. Traps
Each trap runs without stopping the notebook, and needs no API.

**Trap 1: comparing two live runs to show the code is the same.** Sampling alone makes them differ. Same code, same incident, three baseline samples of v0:

In [11]:
for s in range(3):
    r = jsonl[("bad_flag-1", s)]
    print(f"bad_flag-1 sample {s}: steps {r['steps']:2} | writes {r['writes']} | pass {r['pass']}")

bad_flag-1 sample 0: steps  8 | writes [['set_flag', 'ranker_v2_1', 'off']] | pass True
bad_flag-1 sample 1: steps 14 | writes [['set_flag', 'ranker_v2_1', 'off']] | pass True
bad_flag-1 sample 2: steps 12 | writes [['set_flag', 'ranker_v2_1', 'off']] | pass True


**Trap 2: reading 9/10 against 7/10 as a win.** If the true pass rate were 70%, how often would 10 runs score 9 or more by luck alone?

In [12]:
from math import comb
p = 0.7
luck = sum(comb(10, k) * p**k * (1 - p)**(10 - k) for k in (9, 10))
print(f"P(9 or 10 of 10 | true rate 70%) = {luck:.1%}  -> one flip = 10 points; at n = 10 that is noise")

P(9 or 10 of 10 | true rate 70%) = 14.9%  -> one flip = 10 points; at n = 10 that is noise


**Trap 3: comparing only the final grade.** Two runs can both pass with different writes. Search the baseline for such pairs:

In [13]:
from collections import defaultdict
by_inc = defaultdict(list)
for r in rows:
    by_inc[r["incident"]].append(r)
pairs = [(i, a["writes"], b["writes"]) for i, rs in by_inc.items() for a in rs for b in rs
         if a["sample"] < b["sample"] and a["pass"] and b["pass"] and a["writes"] != b["writes"]]
print(len(pairs), "pairs of passing runs with different writes, e.g.")
print(pairs[0])

28 pairs of passing runs with different writes, e.g.
('bad_deploy-0', [['rollback_deploy', 'checkout', None], ['rollback_deploy', 'checkout', None]], [['rollback_deploy', 'checkout', None]])


**Trap 4: a fake model that quietly returns a default when it runs out.** Make it raise, so a mismatch cannot hide.

In [14]:
class Quiet(Scripted):
    def _generate(self, messages, stop=None, run_manager=None, **kw):
        if not self.script:
            return ChatResult(generations=[ChatGeneration(message=AIMessage(content=""))])   # hides the problem
        return super()._generate(messages, stop, run_manager, **kw)

inc, s, v0, replies = next(e for e in EXACT if e[0].id == "memory_leak-5")
try:
    ca = nw_agent.run(inc, model=Scripted(script=[to_message(o) for o in replies]))
except IndexError as e:
    print("Scripted raises:", e)
ca = nw_agent.run(inc, model=Quiet(script=[to_message(o) for o in replies]))
print("Quiet ends with no error | stop:", ca["stop_reason"], "| steps", ca["steps"], "| same trajectory:", same_traj(ca, v0))

Scripted raises: script ran out: create_agent asked for more model calls than v0 made


Quiet ends with no error | stop: model call limit | steps 14 | same trajectory: True


The quiet model makes the run look **identical** to v0, even though `create_agent` asked for a reply v0 never got. With a default instead of a raise, this difference would never have been counted.

## 9. Exercises
Each exercise has a starter cell (`# YOUR CODE HERE`) and a check cell. The checks need no API.

### Exercise 1 (the video's exercise): count every retry toward the 14

Write `CountRetries`, a `ModelCallLimitMiddleware` that adds the retries so far (`len(self.box["errors"])`) to the run's
model-call count before it checks the limit. `run_counted` (given) swaps it in for the normal limit. Then re-run the 39
and count the matches. (Hint: override `before_model(self, state, runtime)`, decorate it with
`@hook_config(can_jump_to=["end"])`, and call `super().before_model` with a changed `run_model_call_count`.)

In [15]:
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware.types import hook_config

class CountRetries(ModelCallLimitMiddleware):
    def __init__(self, box):
        super().__init__(run_limit=14, exit_behavior="end")
        self.box = box
    # YOUR CODE HERE

In [16]:
def run_counted(inc, model):
    orig = nw_agent.make_middleware
    def patched(box, *a, **k):
        mw = orig(box, *a, **k)
        mw[1] = CountRetries(box)                 # slot 1 is ModelCallLimitMiddleware
        return mw
    nw_agent.make_middleware = patched
    try:
        return nw_agent.run(inc, model=model)
    finally:
        nw_agent.make_middleware = orig

try:
    n1, t1, g1 = prove(run_counted)
    print(f"counted retries: identical trajectory {t1}/{n1} | same grade {g1}/{n1}")
    assert t1 >= 37, f"{t1}/39 identical; counting retries should fix at least 3 of the 5"
    print("✓ Exercise 1 passed")
except AssertionError as e:
    print("✗ Exercise 1 not yet:", e)

  differs: memory_leak-5 s2 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out
  differs: traffic_spike-0 s0 v0 steps 14 stop step limit malformed replies 2 | create_agent: script ran out


  differs: traffic_spike-1 s0 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out
  differs: traffic_spike-1 s2 v0 steps 14 stop step limit malformed replies 1 | create_agent: script ran out


  differs: stuck_consumer-5 s1 v0 steps 14 stop step limit malformed replies 3 | create_agent: script ran out


counted retries: identical trajectory 34/39 | same grade 34/39
✗ Exercise 1 not yet: 34/39 identical; counting retries should fix at least 3 of the 5


### Exercise 2: why do two still differ?

Write `malformed_at(replies)`: the 1-based positions of the malformed replies in a list of raw replies. Then look at the
runs that still differ in Exercise 1: where is the last malformed reply?

In [17]:
def malformed_at(replies):
    # YOUR CODE HERE
    return []

In [18]:
try:
    get = {(i.id, s): rep for i, s, _, rep in EXACT}
    assert malformed_at(get[("memory_leak-5", 2)]) == [3], malformed_at(get[("memory_leak-5", 2)])
    assert malformed_at(get[("traffic_spike-1", 2)]) == [14]
    assert malformed_at(get[("stuck_consumer-5", 1)]) == [9, 11, 14]
    print("the two left both end on a malformed 14th reply: v0 stops there, create_agent retries it inside the call")
    print("✓ Exercise 2 passed")
except AssertionError as e:
    print("✗ Exercise 2 not yet:", e)

✗ Exercise 2 not yet: []


### Exercise 3: where do two paths split?

Write `split_at(a, b)`: the 0-based index of the first tool name where two tool lists differ, or `None` if they are equal
(if one is a prefix of the other, the index is the shorter length). Test it on bad_flag-1 from the live run.

In [19]:
def split_at(a, b):
    # YOUR CODE HERE
    return None

In [20]:
try:
    v0_bf = ['check_health', 'read_logs', 'list_flags', 'set_flag', 'check_health', 'finish']
    ca_bf = ['check_health', 'read_logs', 'list_flags', 'set_flag', 'check_health', 'check_health', 'finish']
    assert split_at(v0_bf, ca_bf) == 5, f"bad_flag-1 splits at index 5, got {split_at(v0_bf, ca_bf)}"
    assert split_at(v0_bf, v0_bf) is None
    assert split_at(["a"], ["a", "b"]) == 1 and split_at(["x"], ["y"]) == 0
    print("✓ Exercise 3 passed")
except AssertionError as e:
    print("✗ Exercise 3 not yet:", e)

✗ Exercise 3 not yet: bad_flag-1 splits at index 5, got None


### Exercise 4: is 9 of 10 really better than 7 of 10?

Write `wilson(k, n, z=1.96)`: the 95% Wilson interval `(low, high)` for k passes out of n. Check whether 7/10 and 9/10
overlap, and compare with the 124/180 baseline runs.

In [21]:
import math

def wilson(k, n, z=1.96):
    # YOUR CODE HERE
    return (0.0, 0.0)

In [22]:
try:
    lo7, hi7 = wilson(7, 10)
    lo9, hi9 = wilson(9, 10)
    assert abs(lo7 - 0.397) < 0.002 and abs(hi7 - 0.892) < 0.002, (lo7, hi7)
    assert abs(lo9 - 0.596) < 0.002 and abs(hi9 - 0.982) < 0.002, (lo9, hi9)
    assert lo9 < hi7, "the intervals should overlap"
    lo, hi = wilson(124, 180)
    assert 0.61 < lo < 0.63 and 0.74 < hi < 0.76, (lo, hi)
    print(f"7/10: {lo7:.0%}-{hi7:.0%} | 9/10: {lo9:.0%}-{hi9:.0%} | 124/180: {lo:.0%}-{hi:.0%}  -> overlap: noise")
    print("✓ Exercise 4 passed")
except AssertionError as e:
    print("✗ Exercise 4 not yet:", e)

✗ Exercise 4 not yet: (0.0, 0.0)


## Solutions (try first!)

In [23]:
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.middleware.types import hook_config

class CountRetries(ModelCallLimitMiddleware):
    def __init__(self, box):
        super().__init__(run_limit=14, exit_behavior="end")
        self.box = box

    @hook_config(can_jump_to=["end"])
    def before_model(self, state, runtime):
        counted = state.get("run_model_call_count", 0) + len(self.box["errors"])
        return super().before_model({**state, "run_model_call_count": counted}, runtime)

In [24]:
def malformed_at(replies):
    return [i + 1 for i, o in enumerate(replies) if "error" in o]

In [25]:
def split_at(a, b):
    for i, (x, y) in enumerate(zip(a, b)):
        if x != y:
            return i
    return None if len(a) == len(b) else min(len(a), len(b))

In [26]:
import math

def wilson(k, n, z=1.96):
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)